# Kimonu on the networks of Figures 1 and 2

Kimonu finds concordant complexes, kinetic modules, absolute concentration robustness (ACR) and ACR of
ratios (ACRR) from exact null spaces, and gives the values these relations take at given rate constants.
This notebook runs every call on the two small networks of the manuscript. Everything is exact rational
arithmetic and takes a few seconds.

In [ ]:
from fractions import Fraction
import numpy as np
from scipy import sparse
import kimonu


def network(reactions):
    """A, Y, N, complex names and species from reactions written as 'C + 2P -> 3P' ('0' is the zero complex).

    cobra adds up the two sides of a reaction (C + 2P -> 3P becomes C -> P), which loses the complexes that
    mass action needs, so small networks are written down directly here; for genome-scale models,
    kimonu.build_crn(model) does the same from a cobra model."""
    def side(s):
        comp = {}
        for term in s.split("+"):
            term = term.strip()
            if term != "0":
                n = len(term) - len(term.lstrip("0123456789"))
                comp[term[n:]] = comp.get(term[n:], 0) + (int(term[:n]) if n else 1)
        return comp
    sides = [tuple(side(s) for s in r.split("->")) for r in reactions]
    species = list(dict.fromkeys(sp for s, p in sides for sp in [*s, *p]))
    complexes = list(dict.fromkeys(tuple(sorted(c.items())) for pair in sides for c in pair))
    A = np.zeros((len(complexes), len(sides)), dtype=int)
    Y = np.zeros((len(species), len(complexes)), dtype=int)
    for j, (s, p) in enumerate(sides):
        A[complexes.index(tuple(sorted(s.items()))), j] -= 1
        A[complexes.index(tuple(sorted(p.items()))), j] += 1
    for c, comp in enumerate(complexes):
        for sp, n in comp:
            Y[species.index(sp), c] = n
    names = [" + ".join((f"{n}{sp}" if n > 1 else sp) for sp, n in c) or "0" for c in complexes]
    return sparse.csr_matrix(A), sparse.csr_matrix(Y), sparse.csr_matrix(Y @ A), names, species

## Figure 1

Fourteen reactions among nine species, no exchange with the outside.

In [ ]:
fig1 = ["A -> B", "B -> A", "B -> C", "C -> B", "C -> D", "D + E -> F", "F -> D + E", "F -> B + G",
        "C + G -> H", "H -> C + G", "H -> C + E", "A + G -> I", "I -> A + G", "I -> A + E"]
A, Y, N, cx, sp = network(fig1)
named = lambda groups: [sorted(cx[c] for c in g) for g in groups if len(g) > 1]

conc = kimonu.concordance_from_matrices(A, N)
print("concordant complexes:", named(conc.modules))
print("balanced complexes:  ", sorted(cx[c] for c in conc.balanced))

**Kinetic modules, ACR and ACRR.** The iterated route proves relations at every positive steady state;
the tangent route reads the tangent space at one steady state and bounds it from above. Here they agree.

In [ ]:
kin = kimonu.kinetic_modules(A, Y, N)
acr = kimonu.acr_acrr(Y, kin.modules, metabolite_ids=sp, known_acr=kin.known_acr)
tan = kimonu.tangent_route(A, Y, N, metabolite_ids=sp)
print("kinetic modules:", named(kin.modules))
print("ACR:", acr.acr, "  ACRR pairs:", acr.acrr)
print("tangent route agrees:", sorted(map(sorted, named(tan.modules))) == sorted(map(sorted, named(kin.modules))),
      tan.acr == acr.acr, tan.acrr == acr.acrr)

**Values at given rate constants.** `relation_values` reruns the iterated route at k and returns each value
exactly; `symbolic_formulas` gives it as a formula in k (small networks only, needs sympy). G is fixed by a
rational function of the rate constants that flux coupling alone does not give, so `binomial_formulas`
has no monomial for it.

In [ ]:
k = list(range(1, 15))
print("G at k = 1..14:", kimonu.relation_values(A, Y, N, k, metabolite_ids=sp).acr)
reactions = [str(j) for j in range(1, 15)]
print("G as a formula:", kimonu.symbolic_formulas(A, Y, N, metabolite_ids=sp, reaction_ids=reactions).acr["G"])
print("monomial from flux coupling:", kimonu.binomial_formulas(A, Y, N, metabolite_ids=sp).acr["G"])

**Rate constants for chosen values.** The routes ran at an interior flux r* (`kin.flux`); at k = r* the
steady state is x = 1. `rate_constants` moves it to any x > 0, so every relation takes the value x gives it.

In [ ]:
x = [Fraction(1)] * len(sp)
x[sp.index("G")] = Fraction(3)
k = kimonu.rate_constants(A, Y, kin.flux, x)
print("G at k = r*/psi(x):", kimonu.relation_values(A, Y, N, k, metabolite_ids=sp).acr)

## Figure 2

Protein X is made and degraded, dimerises into an inactive kinase C, and active P activates C cooperatively.
At these rate constants there are two steady states, (X, C, P) = (10, 1, 4) and (10, 4, 1).

In [ ]:
fig2 = ["0 -> X", "X -> 0", "2X -> C", "C -> 0", "C + 2P -> 3P", "P -> 0"]
K = [20, 1, Fraction(1, 20), 1, Fraction(1, 4), 1]
A, Y, N, cx, sp = network(fig2)
named = lambda groups: [sorted(cx[c] for c in g) for g in groups if len(g) > 1]

Ys, S = Y.toarray(), -np.minimum(A.toarray(), 0)
for state in ({"X": 10, "C": 1, "P": 4}, {"X": 10, "C": 4, "P": 1}):
    x = [Fraction(state[s]) for s in sp]
    rates = [K[j] * np.prod([x[i] ** int(Ys[i, c]) for i in range(len(sp)) for c in [S[:, j].argmax()]])
             for j in range(len(fig2))]
    print(state, "dx/dt =", [str(sum(int(N[i, j]) * rates[j] for j in range(len(fig2)))) for i in range(len(sp))])

**The two routes part.** The iterated route proves only the module {P, C + 2P}. The tangent route sees no
direction to move in at either steady state and calls X, C and P fixed (local ACR), but C and P differ
between the two steady states: they are false couplings. X is truly fixed, and the iterated route misses it.

In [ ]:
kin = kimonu.kinetic_modules(A, Y, N, detach_uptakes=False)
tan = kimonu.tangent_route(A, Y, N, metabolite_ids=sp, detach_uptakes=False)
print("iterated: modules", named(kin.modules), " ACR", [sp[i] for i in kin.known_acr])
print("tangent:  modules", named(tan.modules), " local ACR", tan.acr, " dim T =", tan.dimension)

**Values.** In the module {C + 2P, P}, ψ(P) / ψ(C + 2P) = 1 / (C·P) = 1/4 at both steady states. It comes from
flux coupling: R5 and R6 carry the same flux, so k5 C P² = k6 P and 1 / (C·P) = k5 / k6.

In [ ]:
reactions = [str(j) for j in range(1, 7)]
v = kimonu.relation_values(A, Y, N, K, detach_uptakes=False)
b = kimonu.binomial_formulas(A, Y, N, detach_uptakes=False)
s = kimonu.symbolic_formulas(A, Y, N, reaction_ids=reactions, detach_uptakes=False)
for (ref, ratios), (_, monomials), (_, formulas) in zip(v.modules, b.modules, s.modules):
    for c in ratios:
        print(f"psi({cx[c]}) / psi({cx[ref]}) = {ratios[c]} = {monomials[c].format(reactions)} = {formulas[c]}")

**Why X is missed.** The balance of X alone, k1 - k2 X - 2 k3 X² = 0, fixes X = (√(k2² + 8 k1 k3) - k2) / 4 k3
= 10 at both steady states. This is the root of a quadratic. Every value the iterated route can derive is
a product of powers of rational functions of k, and this one is not.

In [ ]:
from math import isqrt
k1, k2, k3 = K[:3]
disc = k2 ** 2 + 8 * k1 * k3
root = isqrt(int(disc))
assert root * root == disc
print("X =", (root - k2) / (4 * k3), "  (k2^2 + 8 k1 k3 =", disc, ")")